# Reproducing Chen et al. (2017) from dandiset 000015

**Paper.** Chen T-W, Li N, Daie K, Svoboda K (2017). A map of anticipatory activity in mouse motor
cortex. *Neuron* 94, 866-879. [doi:10.1016/j.neuron.2017.05.005](https://doi.org/10.1016/j.neuron.2017.05.005).
The claims file (`claims/000015/10.1016-j.neuron.2017.05.005.yaml`) was written from the version of
record, main text, figure legends and STAR Methods; the Supplemental Information was not in the text
read.

**Dataset.** [DANDI:000015](https://dandiarchive.org/dandiset/000015/0.220126.1914), version
0.220126.1914 ([doi:10.48324/dandi.000015/0.220126.1914](https://doi.org/10.48324/dandi.000015/0.220126.1914)).
The paper cites no DANDI version: it names a CRCNS deposit (10.6080/K0S180F9), and that DOI
resolves to a placeholder record ("CRCNS.org ssc-8TBD", "This data is not yet available"). This is
the only published version of the dandiset; the draft has the same 210 assets and size, so the
published version is pinned. The files are streamed; nothing is downloaded.

**Outside data.** The paper compares its imaging with tagged PT and IT units from Li et al. (2015).
Those units are on DANDI:000010, version 0.220126.1905, which is used only for the link claim below.

**What the dandiset holds.** Two-photon imaging of 105 image planes (each deposited twice) from six
mice, with the trial table, the sample, delay and go times, ROI traces and neuropil traces, an ROI
table with PT, IT or unknown labels, and an area label (left ALM or left vM1, the paper's MM) per
plane. It holds no plane depth, no stereotactic coordinates, no lick times, no whisker video, no
wide-field imaging, no silicon probe recordings and no photoinhibition data.

| Claim | Tested here |
|---|---|
| early-lick-fraction, imaging-sample, trials-per-type, neuron-counts | Tested |
| fraction-selective-alm-mm, figs5-single-coefficient | Tested |
| fig4a-lick-direction-epochs, fig4b-object-location-memory, fig4c-outcome-after-lick, fig5-mixed-response-epoch | Tested |
| fig6-object-location-mm, fig6-lick-direction-spread (without its depth part), fig6-outcome-response-uniform, fig6h-preparatory-alm-object-mm | Tested |
| figs4-peak-shift | Tested |
| fig8c-pt-deep-only (counts only), fig8d-pt-contra-bias, fig8d-pt-earlier-than-it (all ALM IT only), fig8d-mm-posterior-preferring, fig8d-error-outnumber-correct, alm-pt-it-lick-fraction-below-20 | Tested |
| li2015-tagged-units-link (link claim added by this attempt) | Tested |
| fig6-lick-direction-deep-alm-first, fig8d-alm-deep-lick-direction, fig8d-mm-object-location, fig8ef-area-explains-variance | Blocked: no plane depth |
| widefield-mm-to-alm, widefield-contralateral-bilateral | Blocked: no wide-field imaging |
| figs9-photoinhibition-alm-mm | Blocked: no photoinhibition data |
| fig1c-whisking-similar | Blocked: no whisker tracking |
| ephys-selective-counts, fig7d-ephys-area-distribution | Blocked: the silicon probe data were never made public |
| ephys-pt-it-onset-not-detected | Blocked: method for the Li et al. (2015) units not fixed (question asked) |

The conventions T, M, A, D, B and O cited below are those of the claims file's opening comment; the
helper module `chen2017.py` says where each parameter comes from.

In [ ]:
import collections
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import chen2017 as C

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = C.list_assets()
headers = C.read_headers(assets)
print(f"read {len(headers)} files of dandiset {C.DANDISET_ID} version {C.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

# Copies of one plane share subject, session, frame and ROI counts, trial table and the first 200
# frames of both trace series; each plane is counted once, as its first path.
groups = collections.defaultdict(list)
for h in headers:
    groups[C.content_key(h)].append(h)
planes = sorted((sorted(v, key=lambda h: h["path"])[0] for v in groups.values()), key=lambda h: h["path"])
results = C.process_planes(planes, shuffle_seed=0, start_events=("sample", "start_time"))
print(f"processed {len(planes)} planes in {time.time() - started:.0f} s")

# One row per ROI.
cells = pd.DataFrame({
    "plane": np.concatenate([[i] * h["n_rois"] for i, h in enumerate(planes)]),
    "animal": np.concatenate([[h["subject"]] * h["n_rois"] for h in planes]),
    "area": np.concatenate([[h["area"]] * h["n_rois"] for h in planes]),
    "cell_type": np.concatenate([h["cell_type"] for h in planes]),
    "included": np.concatenate([h["included"] for h in planes]),
    "f0_min": np.concatenate([r["T1"]["f0_min"] for r in results]),
    "f0_nonpositive": np.concatenate([r["T1"]["f0_nonpositive"] for r in results]),
    "dff_abs_max": np.concatenate([r["T1"]["dff_abs_max"] for r in results]),
})
area, ctype = cells.area.values, cells.cell_type.values
animal, plane = cells.animal.values, cells.plane.values
ALM, MM = area == "ALM", area == "MM"
A = {b: C.assemble(planes, results, b) for b in ("T1", "T2")}

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{k: h[k] for k in ("path", "identifier", "subject", "session", "area", "location",
                                         "rate", "n_frames", "n_rois", "size")} for h in headers])
info["copies"] = [len(groups[C.content_key(h)]) for h in headers]
print("Files:", len(headers), " distinct planes:", len(planes),
      " copies per plane:", dict(collections.Counter(len(v) for v in groups.values())))
print("Distinct SHA-256 digests over the 210 files:", len({h['sha256'] for h in headers}),
      "; distinct identifiers:", info.identifier.nunique())
pairs = [sorted(h["identifier"] for h in v) for v in groups.values()]
same_size = all(len({h["size"] for h in v}) == 1 for v in groups.values())
print("Every pair has the same file size:", same_size)
print("First five pairs (identifiers):", pairs[:5])
print("\nPlanes by subject and area (the area is from experiment_description; ImagingPlane.location):")
pl = pd.DataFrame([{k: h[k] for k in ("subject", "session", "area", "location", "rate", "n_rois")} for h in planes])
print(pd.crosstab(pl.subject, pl.area, margins=True).to_string())
print("\nImagingPlane.location values:", dict(collections.Counter(pl.location)))
print("Imaging rates (Hz), planes:", dict(collections.Counter(pl.rate.round(4))))
print("Rate in the trace series equals the plane's imaging_rate:", all(abs(h["rate"] - h["rate_attr"]) < 1e-9 for h in planes),
      "; starting_time values:", sorted({h["starting_time"] for h in planes}))
print("Units of the trace series:", sorted({str(h["rois_unit"]) for h in planes}))
print("Sessions (subject and date):", pl.groupby(["subject", "session"]).ngroups)

Every plane is deposited twice: 105 pairs of files with the same size, the same trial table and the
same first 200 frames of both trace series, but different identifiers (for example
`an041_2014-08-23_21` and `an041_2014-08-23_43`) and different digests. A full comparison of two
pairs while preparing this notebook (not run here) found every dataset identical except `identifier`
and `file_create_date`. Each plane is counted once. `ImagingPlane.location` is "ALM" in every file,
also in the 47 planes whose `experiment_description` says "left_vm1"; the area used here (convention
A, reading A1) is from `experiment_description`, and its counts are checked against the paper's
below. The frame rates (13.95 and 14.20 Hz) agree with the stated 14 Hz. The traces are in "a.u.".

In [ ]:
tt_all = np.concatenate([C.trial_types(h) for h in planes])
early = np.concatenate([h["early_lick"] for h in planes])
outc = np.concatenate([h["outcome"] for h in planes])
instr = np.concatenate([h["instruction"] for h in planes])
print("Trials in all planes:", len(outc))
print("outcome x early_lick:")
print(pd.crosstab(outc, early).to_string())
print("\ntrial_instruction x early_lick:")
print(pd.crosstab(instr, early).to_string())
print("\nTrial types (convention T):", dict(collections.Counter(tt_all)))
print("task values:", sorted({t for h in planes for t in h["task"]}))

rows = []
for h in planes:
    ok = C.trial_types(h) != ""
    rows.append({"go_minus_sample": h["go"][ok] - h["sample"][ok], "delay_minus_sample": h["delay"][ok] - h["sample"][ok],
                 "sample_minus_start": h["sample"] - h["trial_start"],
                 "frames_s_minus_last_stop": h["n_frames"] / h["rate"] - h["trial_stop"][-1]})
gs = np.concatenate([r["go_minus_sample"] for r in rows])
ds = np.concatenate([r["delay_minus_sample"] for r in rows])
ss = np.concatenate([r["sample_minus_start"] for r in rows])
fl = np.array([r["frames_s_minus_last_stop"] for r in rows])
print(f"\nAnalysed trials: go - sample {gs.min():.3f} to {gs.max():.3f} s (median {np.median(gs):.3f}); "
      f"delay - sample {ds.min():.3f} to {ds.max():.3f} s")
print(f"All trials: sample - start_time {ss.min():.3f} to {ss.max():.3f} s")
print(f"Recording length (frames / rate) minus the last trial's stop_time: {fl.min():.3f} to {fl.max():.3f} s")
print("Analysed trials dropped because the window leaves the recording:", sum(r["T1"]["dropped_edge"] for r in results))

The trial table holds 'hit', 'miss', 'ignore' and 'non-performing' outcomes. Every early-lick trial
has outcome and instruction 'non-performing', so its pole position is not recorded; 'ignore' trials
(no lick after the go cue) are left out as convention T says. On every analysed trial the go cue
comes 4.2 s after the `sample` event and the delay starts 1.2 s after it, as the paper's sample (1.2 s)
and delay (3 s) epochs require, while the trial table's `start_time` is 0.623 s before `sample`.
So the paper's trial start, at which the claims file puts the go cue 4.2 s earlier, is the `sample`
event, and it is used as trial start throughout. The traces cover the trial table to within a
frame. A few analysed trials whose window runs past the end of the recording are dropped (count
above).

In [ ]:
print("ROIs:", len(cells), " by area:", dict(collections.Counter(area)))
print("cell_type values by area:")
print(pd.crosstab(cells.area, cells.cell_type, margins=True).to_string())
print("\ncell_type by animal:")
print(pd.crosstab(cells.animal, cells.cell_type).to_string())
print("\n'included' (described as 'whether to include this roi into later analyses'):",
      dict(collections.Counter(cells.included)), " by area:", cells.groupby("area").included.sum().to_dict())
with C._open(planes[0]["asset_id"]) as fh:
    names = []
    fh.visit(lambda n: names.append(n) if not n.startswith("specifications") else None)
    ip_fields = sorted(fh["general/optophysiology/ImagingPlane"].keys())
print("\nObjects in one file, to three levels (except the NWB specification):")
print([n for n in names if n.count("/") <= 2])
print("ImagingPlane fields:", ip_fields)

The ROI counts are those of the paper (checked under neuron-counts). PT and IT labels are present in
three mice (an063, an064, an066), as the paper says for its retrograde labelling. The `included`
column marks 8,655 of 15,431 ROIs, but the paper's counts use all ROIs and nothing says what
excluded the others; all ROIs are used here, and the included ROIs are shown as a variant where the
counts matter. The files hold no plane depth (the ImagingPlane has no depth, origin or coordinates),
no lick times, no whisker data and no other recordings, so the claims that need these are blocked.

In [ ]:
roi_trials = sum(h["n_rois"] * r["T1"]["n_trials"] for h, r in zip(planes, results))
print(f"ROI x trial baselines F0 <= 0: {int(cells.f0_nonpositive.sum())} of {roi_trials} "
      f"({100 * cells.f0_nonpositive.sum() / roi_trials:.2f}%)")
print("ROIs with F0 <= 0 on at least one analysed trial:", int((cells.f0_min <= 0).sum()),
      " by area:", cells[cells.f0_min <= 0].groupby("area").size().to_dict())
print("Largest |dF/F| per ROI over trials and bins, percentiles 50, 90, 99, 99.9:",
      np.percentile(cells.dff_abs_max, [50, 90, 99, 99.9]).round(1))
print(f"Median over ROIs of the per-ROI median F0: {np.median(np.concatenate([r['T1']['f0_median'] for r in results])):.2f} a.u.")

After neuropil subtraction (r = 0.7) the baseline F0 is small, and for 0.26% of ROI-trials it is zero
or negative, in 1,262 ROIs. dF/F = (F - F0)/F0 then reaches thousands. Neither the claims file nor the
paper says how such trials are treated. The analysis follows the formula as given, and a question was
asked; the claims whose numbers are means of dF/F or of coefficients also show the result without
the ROIs that have F0 <= 0 on any trial, as a variant.

**Clock check.** The ROI traces and the trial table share one clock if trial-type selectivity
appears only after the task events that cause it. Below, per bin, the fraction of ROIs with a
significant object-location (alpha) or lick-direction (beta) factor, with the true trial types,
with the trial types permuted within each plane (the null), and with the trials aligned to the
trial table's `start_time` in place of `sample`.

In [ ]:
g = A["T1"]["times"]
E = A["T1"]["epochs"]
sig_true = A["T1"]["sig"]
sig_shuf = A["T1"]["shuffled"]["sig"]
p_start = np.concatenate([r["T1_start_time"]["p"] for r in results])
sig_start = p_start < C.ALPHA
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for k, name in enumerate(["object location (alpha)", "lick direction (beta)"]):
    ax = axes[k]
    for s, lab in ((sig_true, "trial types, aligned to sample"), (sig_shuf, "permuted trial types"),
                   (sig_start, "aligned to start_time")):
        ax.plot(g, s[..., k].mean(0), label=lab)
    for x in (0, 1.2, 4.2):
        ax.axvline(x, color="k", lw=0.5)
    ax.set_title(name); ax.set_xlabel("time from trial start (s)")
axes[0].set_ylabel("fraction of ROIs, p < 0.01"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()
for k, name in enumerate(["alpha", "beta"]):
    ft, fs, fa = sig_true[..., k].mean(0), sig_shuf[..., k].mean(0), sig_start[..., k].mean(0)
    print(f"{name}: true, before trial start {ft[E['pre']].mean():.4f}; max {ft.max():.4f} at {g[ft.argmax()]:.2f} s | "
          f"permuted, range {fs.min():.4f} to {fs.max():.4f} | aligned to start_time, max {fa.max():.4f} at {g[fa.argmax()]:.2f} s")
first_a = g[np.flatnonzero(sig_true[..., 0].mean(0) > 0.02)[0]]
first_a_start = g[np.flatnonzero(sig_start[..., 0].mean(0) > 0.02)[0]]
print(f"First bin where alpha-significant ROIs exceed 2%: {first_a:.2f} s aligned to sample, "
      f"{first_a_start:.2f} s aligned to start_time (difference {first_a_start - first_a:.2f} s)")

Before trial start the fraction of ROIs with a significant factor is at the 1% the test allows, and
with permuted trial types it stays near 1% in every bin. With the true trial types, object-location
selectivity rises after the pole starts to move and lick-direction selectivity after the go cue;
aligned to `start_time` the same rise comes later on the time axis, as the offset between the
two events predicts (0.65 s for the first bin above 2%, against the 0.623 s offset, within a
frame). The traces and the trial table are consistent with one clock; the check shows
the alignment to within a few frames, not exactly.

## early-lick-fraction

> Mice licked early, during the delay epoch, on 13.3% +- 10% of trials, and these trials were not
> analysed. (Expected 13.3, tolerance 10, percent.)

In [ ]:
rows = []
for h in planes:
    n = len(h["outcome"])
    rows.append({"subject": h["subject"], "session": h["session"], "n": n, "early": int(h["early_lick"].sum()),
                 "with_lick": n - int((h["outcome"] == "ignore").sum())})
el = pd.DataFrame(rows)
out = []
for den in ("n", "with_lick"):
    per_plane = 100 * el.early / el[den]
    ses = el.groupby(["subject", "session"])[["early", den]].sum()
    mouse = el.groupby("subject")[["early", den]].sum()
    out += [
        {"reading": "(a) mean over planes", "denominator": den, "mean": per_plane.mean(), "sd": per_plane.std()},
        {"reading": "(b) mean over sessions", "denominator": den, "mean": (100 * ses.early / ses[den]).mean(), "sd": (100 * ses.early / ses[den]).std()},
        {"reading": "(c) mean over mice", "denominator": den, "mean": (100 * mouse.early / mouse[den]).mean(), "sd": (100 * mouse.early / mouse[den]).std()},
        {"reading": "(d) pooled", "denominator": den, "mean": 100 * el.early.sum() / el[den].sum(), "sd": np.nan}]
res = pd.DataFrame(out)
res["within_13.3_pm_10"] = (res["mean"] - 13.3).abs() <= 10
print(res.round(2).to_string(index=False))
print("\nPer mouse (percent of all trials):", (100 * el.groupby("subject").early.sum() / el.groupby("subject").n.sum()).round(1).to_dict())

**Result: reproduced.** Early-lick trials are 13.24% +- 9.14% of trials, averaged over planes with all
trials as denominator (reading a), within 13.3 +- 10; every other reading is within tolerance too
(table). They are excluded from the analysis.

## imaging-sample

> Two-photon imaging covered 12 locations in 6 mice with 8.8 +- 2.6 planes per location, 105 planes in
> all, each imaged for 92 +- 26 trials at 73% +- 9% correct, 3 +- 1 planes per session.

In [ ]:
pp = pd.DataFrame([{"subject": h["subject"], "session": h["session"], "area": h["area"],
                    "n_all": len(h["outcome"]), "n_four": int((C.trial_types(h) != "").sum()),
                    "hits_four": int(np.isin(C.trial_types(h), ["CR", "CL"]).sum()),
                    "hits_all": int((h["outcome"] == "hit").sum())} for h in planes])
print("mice:", pp.subject.nunique(), " planes:", len(pp))
print("subject x area combinations (no location identifier is in the files):", pp.groupby(["subject", "area"]).ngroups)
print(pp.groupby(["subject", "area"]).size().to_string())
print(f"\ntrials per plane (a) four types: {pp.n_four.mean():.2f} +- {pp.n_four.std():.2f};  (b) all trials: {pp.n_all.mean():.2f} +- {pp.n_all.std():.2f}")
pc_a = 100 * pp.hits_four / pp.n_four
pc_b = 100 * pp.hits_all / pp.n_all
print(f"percent correct per plane (a) of the four types: {pc_a.mean():.2f} +- {pc_a.std():.2f};  (b) of all trials: {pc_b.mean():.2f} +- {pc_b.std():.2f}")
ps = pp.groupby(["subject", "session"]).size()
print(f"planes per session: {ps.mean():.2f} +- {ps.std():.2f}, range {ps.min()} to {ps.max()}, sessions {len(ps)}")
print("\nper mouse: planes, mean trials (a), mean percent correct (a)")
print(pp.assign(pc=pc_a).groupby("subject").agg(planes=("n_four", "size"), trials=("n_four", "mean"), pc=("pc", "mean")).round(1).to_string())

**Result: partially reproduced.** Six mice and 105 planes, as stated. With the four analysed trial
types (reading a), planes have 92.15 +- 25.98 trials at 73.83% +- 9.03% correct, both within
tolerance; with all trials (reading b) there are 107.37 trials per plane, within tolerance, at 63.89%
correct, just outside it. Planes per session are 3.09 +- 1.11 (range 1 to 6, 34 sessions). The 12
locations and the 8.8 planes per location cannot be counted: the files carry no location
identifier, and subject and area give only 10 combinations.

## trials-per-type

> Each imaged plane had on average 34 +- 13 CR, 36 +- 14 CL, 12 +- 5 ER and 10 +- 4 EL trials.

In [ ]:
tc = pd.DataFrame([collections.Counter(C.trial_types(h)) for h in planes]).fillna(0)[list(C.TYPES)]
exp = {"CR": (34, 13), "CL": (36, 14), "ER": (12, 5), "EL": (10, 4)}
for t in C.TYPES:
    print(f"{t}: {tc[t].mean():.2f} +- {tc[t].std():.2f} (range {int(tc[t].min())} to {int(tc[t].max())}); "
          f"expected {exp[t][0]} +- {exp[t][1]}; within: {abs(tc[t].mean() - exp[t][0]) <= exp[t][1]}")
print("Planes with fewer than 2 trials of some type:", int((tc.min(axis=1) < 2).sum()))

**Result: reproduced.** Mean trials per plane are CR 34.06 +- 13.43, CL 35.51 +- 13.94, ER 12.18 +- 4.71
and EL 10.40 +- 4.42, each within the paper's spread (34 +- 13, 36 +- 14, 12 +- 5, 10 +- 4), and the
spreads match too.

## neuron-counts

> 15,431 neurons were imaged across 6 mice, 8,837 in ALM and 6,594 in MM.

In [ ]:
n_alm, n_mm = int(ALM.sum()), int(MM.sum())
print("ROIs:", len(cells), " ALM (reading A1, experiment_description left_alm):", n_alm, " MM (left_vm1):", n_mm)
print("Reading A2 (nearer area centre) cannot be run: no ROI or plane coordinates are in the files.")
print("per mouse:", cells.groupby(["animal", "area"]).size().to_dict())

**Result: reproduced.** 15,431 ROIs, 8,837 in ALM and 6,594 in MM by the files' area label (reading
A1), exactly the paper's numbers. Reading A2 cannot be run; A1 is used by every claim that splits
ALM and MM.

## fraction-selective-alm-mm

> A large fraction of neurons distinguished between trial types, 23% in MM (1,523 of 6,594) and 31% in
> ALM (2,754 of 8,837).

Readings from the criterion: rule R1 (one significant bin) or R2 (two consecutive), window W1 (4.6 s
before to 2.5 s after the go cue), W2 (the whole extracted window) or W3 (trial start to 2.5 s after
the go cue), bins T1 or T2, area A1.

In [ ]:
def selective_neurons(anysel, times, rule, window):
    w = {"W1": (times >= -0.4 - 1e-9) & (times <= 6.7 + 1e-9), "W2": np.ones(len(times), bool),
         "W3": (times >= 0) & (times <= 6.7 + 1e-9)}[window]
    a = anysel[:, w]
    return a.any(1) if rule == "R1" else (a[:, 1:] & a[:, :-1]).any(1)

rows = []
for b in ("T1", "T2"):
    for rule in ("R1", "R2"):
        for win in ("W1", "W2", "W3"):
            s = selective_neurons(A[b]["any"], A[b]["times"], rule, win)
            r = {"bins": b, "rule": rule, "window": win, "ALM_n": int(s[ALM].sum()), "ALM_%": 100 * s[ALM].mean(),
                 "MM_n": int(s[MM].sum()), "MM_%": 100 * s[MM].mean()}
            r["all_within"] = (abs(r["MM_n"] - 1523) <= 152 and abs(r["ALM_n"] - 2754) <= 275 and
                               abs(r["MM_%"] - 23) <= 2.3 and abs(r["ALM_%"] - 31) <= 3.1)
            r["ALM_higher"] = r["ALM_%"] > r["MM_%"]
            if b == "T1":
                sn = selective_neurons(A[b]["shuffled"]["any"], A[b]["times"], rule, win)
                r["null_ALM_%"], r["null_MM_%"] = 100 * sn[ALM].mean(), 100 * sn[MM].mean()
            rows.append(r)
fs = pd.DataFrame(rows)
print(fs.round(1).to_string(index=False))
print("\nReadings with all four values within tolerance:", int(fs.all_within.sum()))

print("\nVariants of R2, W1, T1 (not deciding):")
s = selective_neurons(A["T1"]["any"], A["T1"]["times"], "R2", "W1")
for name, m in (("all ROIs", np.ones(len(cells), bool)), ("ROIs with F0 > 0 on every trial", cells.f0_min.values > 0),
                ("ROIs marked included", cells.included.values)):
    print(f"  {name}: ALM {int(s[ALM & m].sum())}/{int((ALM & m).sum())} ({100 * s[ALM & m].mean():.1f}%), "
          f"MM {int(s[MM & m].sum())}/{int((MM & m).sum())} ({100 * s[MM & m].mean():.1f}%)")
print("\nR2, W1, T1 by animal (percent selective):")
print(pd.DataFrame({"sel": s, "animal": animal, "area": area}).groupby(["animal", "area"]).sel.mean().mul(100).round(1).unstack().to_string())
print("\nPer-bin fraction of ROIs selective for any factor, T1: before trial start "
      f"{A['T1']['any'][:, E['pre']].mean():.4f}, max ALM {A['T1']['any'][ALM].mean(0).max():.4f}, max MM {A['T1']['any'][MM].mean(0).max():.4f}")

**Result: not reproduced.** No reading puts all four values within tolerance. Rule R1 marks 78% to 91%
of ROIs as selective, and 72% to 85% with permuted trial types, so it counts chance bins. Rule R2
(permuted: 10.5% to 14.3%) with T1 bins comes closest: ALM 2,877 (32.6%) and MM 1,755 (26.6%) in window W1; ALM is
within tolerance, MM is above it (at most 1,675 and 25.3%). With T2 bins R2 gives 45% and 38%. ALM
has the higher fraction under every reading. Leaving out ROIs with F0 <= 0 gives MM 1,675 of 5,854
(28.6%), still above the tolerance on the fraction.

## figs5-single-coefficient

> The selectivity of most neurons can be attributed to one of the three coefficients, especially
> early in the trial, so most neurons mainly encode one behavioural variable at a time.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; Eb = A[b]["epochs"]
    n_any = A[b]["any"].sum(0)
    frac_pure = A[b]["pure"].any(-1).sum(0) / np.maximum(n_any, 1)
    w = (t >= 0) & (t < C.GO)
    print(f"{b}: bins from trial start to the go cue {w.sum()}, with pure fraction > 0.5: {(frac_pure[w] > 0.5).sum()}; "
          f"range {frac_pure[w].min():.3f} to {frac_pure[w].max():.3f}; selective ROIs per bin {n_any[w].min()} to {n_any[w].max()}; "
          f"after the go cue mean {frac_pure[Eb['response']].mean():.3f} (min {frac_pure[Eb['response']].min():.3f})")
    if b == "T1":
        plt.figure(figsize=(5, 2.6)); plt.plot(t, frac_pure); plt.axhline(0.5, color="k", lw=0.5)
        plt.xlabel("time from trial start (s)"); plt.ylabel("pure / selective"); plt.title("T1"); plt.show()

**Result: reproduced.** In every bin from trial start to the go cue, more than half of the
selective ROIs are purely selective (lowest 0.845 with T1 bins, 0.826 with T2). After the go cue the
fraction is 0.793 (T1) on average.

## fig4a-lick-direction-epochs

> Different neurons coded for lick direction in different epochs, some during the sample and delay
> epochs before licking, some during the response epoch, and some after the mouse stopped consuming
> water.

In [ ]:
def fig4a_counts(pure, t):
    pb = pure[..., 1]
    first = np.where(pb.any(1), np.argmax(pb, 1), -1)
    go_bin = np.flatnonzero(t >= C.GO)[0]
    return {"delay": int(pb[:, (t >= C.SAMPLE_END) & (t < C.GO)].any(1).sum()),
            "first_after_go": int((first >= go_bin).sum()),
            "after_go_plus_2s": int(pb[:, t > C.GO + 2].any(1).sum())}
for b in ("T1", "T2"):
    print(b, fig4a_counts(A[b]["pure"], A[b]["times"]))
print("T1 with permuted trial types (null):", fig4a_counts(A["T1"]["shuffled"]["pure"], A["T1"]["times"]))
print("Lick times are not in the files, so the median last lick cannot be computed.")

**Result: reproduced.** With T1 bins, 4,125 ROIs are purely lick-direction selective in the delay
epoch, 3,611 are first so after the go cue, and 4,426 more than 2 s after the go cue (T2: 3,825,
3,514 and 4,155). With permuted trial types the same counts are 3,273, 2,526 and 3,196, so each group
exceeds chance, but most of each count is what chance bins alone give. The time after the last
lick cannot be tested: lick times are not in the files.

## fig4b-object-location-memory

> Many object-location-selective neurons were selective during the sample epoch, and a subset coded
> object location mainly in the delay epoch after the pole was removed, some up to the end of the delay.

In [ ]:
def fig4b_counts(pure, t):
    pa = pure[..., 0]
    d = pa[:, (t >= C.SAMPLE_END) & (t < C.GO)].sum(1); tot = pa.sum(1)
    return {"sample": int(pa[:, (t >= 0) & (t < C.SAMPLE_END)].any(1).sum()),
            "most_bins_in_delay": int(((tot > 0) & (d > tot / 2)).sum()),
            "last_0.5s_before_go": int(pa[:, (t >= C.GO - 0.5) & (t < C.GO)].any(1).sum())}
for b in ("T1", "T2"):
    print(b, fig4b_counts(A[b]["pure"], A[b]["times"]))
print("T1 with permuted trial types (null):", fig4b_counts(A["T1"]["shuffled"]["pure"], A["T1"]["times"]))

**Result: reproduced.** With T1 bins, 2,148 ROIs are purely object-location selective in the sample
epoch, 2,487 have most of their object-location bins in the delay, and 1,149 are selective in the
last 0.5 s before the go cue (T2: 2,067, 2,483, 1,074). With permuted trial types the counts are
1,694, 1,719 and 707, so each group exceeds chance.

## fig4c-outcome-after-lick

> Outcome-selective neurons typically became selective after the answer lick; some preferred correct
> and others error trials, and some kept the outcome long after the trial ended.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; pg = A[b]["pure"][..., 2]; K = A[b]["coef"][..., 2]
    ever = pg.any(1); first = np.argmax(pg, 1); go_bin = np.flatnonzero(t >= C.GO)[0]
    after = ever & (first >= go_bin)
    sign_first = np.sign(K[np.arange(len(K)), first])[ever]
    print(f"{b}: purely outcome-selective ROIs {ever.sum()}, first bin after the go cue {after.sum()} "
          f"({after.sum() / ever.sum():.3f}); gamma sign at the first bin: correct-preferring {(sign_first > 0).sum()}, "
          f"error-preferring {(sign_first < 0).sum()}; latest outcome-selective bin {t[np.flatnonzero(pg.any(0))[-1]]:.2f} s")
    if b == "T1":
        s2 = (pg[:, 1:] & pg[:, :-1])
        ever2 = s2.any(1); first2 = np.argmax(s2, 1)
        print(f"   variant, two consecutive bins (as rule R2): {ever2.sum()} ROIs, first after the go cue "
              f"{(ever2 & (first2 >= go_bin)).sum() / ever2.sum():.3f}")
pgs = A["T1"]["shuffled"]["pure"][..., 2]
evs = pgs.any(1)
print(f"T1 permuted (null): {evs.sum()} ROIs, first after the go cue {(evs & (np.argmax(pgs, 1) >= go_bin)).sum() / evs.sum():.3f}")

**Result: not reproduced.** Of the ROIs purely outcome selective in any bin (9,431 with T1 bins), 41.3%
have their first such bin after the go cue (T2 41.8%), below the half the criterion requires. Most
of these ROIs have only chance bins: with permuted trial types 8,070 ROIs qualify, 29.6% of them
first after the go cue. Requiring two consecutive bins (a variant, not deciding) leaves 1,463 ROIs,
78.5% of them first after the go cue. Both signs of gamma occur (4,157 correct-preferring and 5,274
error-preferring at the first bin), and outcome selectivity lasts to the end of the window (9.17 s
after trial start).

## fig5-mixed-response-epoch

> Most neurons with mixed selectivity were seen during the response epoch; a subset responded in only
> one trial type and a few switched the variable they coded during the trial.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; mixed = A[b]["mixed"]; pure = A[b]["pure"]; Eb = A[b]["epochs"]
    nb = mixed.sum(0)
    early_obj = pure[:, t < C.GO, 0].any(1)
    late_other = (pure[:, t >= C.GO, 1] | pure[:, t >= C.GO, 2]).any(1)
    print(f"{b}: mixed ROI-bins {nb.sum()}, after the go cue {nb[Eb['response']].sum()} ({nb[Eb['response']].sum() / nb.sum():.3f}); "
          f"ROIs purely object-location selective before the go cue and purely lick-direction or outcome selective after it: "
          f"{(early_obj & late_other).sum()}")
nbs = A["T1"]["shuffled"]["mixed"].sum(0)
print(f"T1 permuted (null): mixed ROI-bins {nbs.sum()}, after the go cue {nbs[E['response']].sum() / nbs.sum():.3f} "
      f"(the response window is {E['response'].mean():.3f} of the bins)")

**Result: reproduced.** 76.5% of mixed-selective ROI-bins fall after the go cue with T1 bins (77.6%
with T2), against 50% of the bins in the window and 52.3% with permuted labels. 4,682 ROIs (T1) are
purely object-location selective before the go cue and purely lick-direction or outcome selective
after it.

## fig6-object-location-mm

> During the sample and early delay epochs, object location was represented mainly in MM.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; pa = A[b]["pure"][..., 0]
    fa, fm = pa[ALM].mean(0), pa[MM].mean(0)
    for end in (2.2, 2.7):
        w = (t >= 0) & (t <= end + 1e-9)
        print(f"{b}, trial start to {end} s: MM above ALM in {(fm[w] > fa[w]).sum()} of {w.sum()} bins; "
              f"mean MM {fm[w].mean():.4f}, ALM {fa[w].mean():.4f}")
t = A["T1"]["times"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
names = ["object location", "lick direction", "outcome"]
for ax, (lab, m) in zip(axes, (("ALM", ALM), ("MM", MM))):
    for k in range(3):
        ax.plot(t, A["T1"]["pure"][m][..., k].mean(0), label=names[k])
    ax.plot(t, A["T1"]["mixed"][m].mean(0), label="mixed")
    for x in (0, 1.2, 4.2):
        ax.axvline(x, color="k", lw=0.5)
    ax.set_title(f"{lab}, T1"); ax.set_xlabel("time from trial start (s)")
axes[0].set_ylabel("fraction of ROIs"); axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()
for lab, m in (("ALM", ALM), ("MM", MM)):
    print(lab, "plotted fractions range:", {n: (round(float(A['T1']['pure'][m][..., k].mean(0).min()), 4), round(float(A['T1']['pure'][m][..., k].mean(0).max()), 4)) for k, n in enumerate(names)},
          "mixed", (round(float(A['T1']['mixed'][m].mean(0).min()), 4), round(float(A['T1']['mixed'][m].mean(0).max()), 4)))
pa = A["T1"]["pure"][..., 0]; w = (t >= 0) & (t <= 2.2 + 1e-9)
print("\nby animal, mean fraction purely object-location selective, trial start to 2.2 s (T1):")
print(pd.DataFrame({"f": pa[:, w].mean(1), "animal": animal, "area": area}).groupby(["animal", "area"]).f.mean().round(4).unstack().to_string())

**Result: reproduced.** The fraction of purely object-location-selective ROIs is higher in MM than in
ALM in 27 of 31 bins from trial start to 2.2 s, and 34 of 38 to 2.7 s, with T1 bins (T2: 29 of 33
and 37 of 41).

## fig6-lick-direction-spread

> Lick-direction selectivity, in number of neurons and strength, kept increasing and spread across ALM
> layers and to a minor extent into MM, peaking after the go cue.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; Eb = A[b]["epochs"]
    pb = A[b]["pure"][..., 1]; ab = np.abs(A[b]["coef_sig"][..., 1])
    fA, fM, bA = pb[ALM].mean(0), pb[MM].mean(0), ab[ALM].mean(0)
    print(f"{b}: (a) ALM fraction peaks at {t[fA.argmax()]:.2f} s ({fA.max():.4f}), ALM mean |beta| peaks at {t[bA.argmax()]:.2f} s "
          f"({bA.max():.4f}); both after the go cue: {t[fA.argmax()] >= C.GO and t[bA.argmax()] >= C.GO}")
    print(f"    (c) MM maximum {fM.max():.4f} at {t[fM.argmax()]:.2f} s; MM in the delay: mean {fM[Eb['delay']].mean():.4f}, "
          f"max {fM[Eb['delay']].max():.4f}; ALM maximum {fA.max():.4f}; "
          f"holds with the delay mean: {fM.max() > fM[Eb['delay']].mean() and fM.max() < fA.max()}, "
          f"with the delay maximum: {fM.max() > fM[Eb['delay']].max() and fM.max() < fA.max()}")
print("(b) superficial ALM cannot be separated: the files hold no plane depth.")
ab = np.abs(A["T1"]["coef_sig"][..., 1]); keep = cells.f0_min.values > 0
print(f"Variant, ROIs with F0 > 0 on every trial (T1): ALM mean |beta| peaks at {t[ab[ALM & keep].mean(0).argmax()]:.2f} s")
print("by animal, time of the ALM peak of the fraction (T1):",
      {a: round(float(t[A['T1']['pure'][ALM & (animal == a)][..., 1].mean(0).argmax()]), 2) for a in np.unique(animal[ALM])})

**Result: partially reproduced.** In ALM the fraction of lick-direction-selective ROIs and their mean
|beta| both peak after the go cue (4.44 and 4.66 s, T1), and in MM the fraction peaks at 0.026, above
its delay-epoch level (mean 0.008, maximum 0.015) and below the ALM peak (0.062), under both
readings of "its value in the delay epoch" and both bin readings. The spread across ALM layers (part
b) cannot be tested, as the files hold no depth.

## fig6-outcome-response-uniform

> Outcome-selective neurons were seen only during the response epoch, scattered roughly uniformly
> through ALM and MM; mixed neurons were also seen mainly in the response epoch in both areas.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; resp = (t >= C.GO) & (t <= C.GO + 2.5 + 1e-9)
    for name, X in (("outcome", A[b]["pure"][..., 2]), ("mixed", A[b]["mixed"])):
        vals = {}
        for lab, m in (("ALM", ALM), ("MM", MM)):
            f = X[m].mean(0)
            vals[lab] = (f[t < C.GO].max(), f[t >= C.GO].max(), f[resp].mean())
        a_ok = all(v[0] < v[1] / 3 for v in vals.values())
        ratio = max(vals["ALM"][2], vals["MM"][2]) / min(vals["ALM"][2], vals["MM"][2])
        print(f"{b} {name}: " + "; ".join(f"{k} max before go {v[0]:.4f}, max after {v[1]:.4f}, response mean {v[2]:.4f}" for k, v in vals.items())
              + f" | (a) {a_ok}; (b) ratio {ratio:.3f}, below 1.5: {ratio < 1.5}")
ps = A["T1"]["shuffled"]["pure"][..., 2]
print(f"T1 permuted (null), outcome fraction per bin: {ps.mean(0).min():.4f} to {ps.mean(0).max():.4f}")

**Result: not reproduced.** Part (b) holds: averaged over the response epoch the ALM and MM outcome
fractions differ by a factor of 1.46 (T1) and 1.45 (T2). Part (a) fails in MM: its largest fraction
before the go cue (0.0112, T1) is not below a third of the largest after it (0.0268), while in ALM it
just is (0.0123 against 0.0373); T2 gives the same. The pre-cue maxima are near the range of the
permuted-label fractions (0.0067 to 0.0105). Mixed ROIs, reported only, are 2.8 times as frequent in
ALM as in MM over the response epoch.

## fig6h-preparatory-alm-object-mm

> Throughout the trial preparatory activity, as the fraction of lick-direction-selective neurons or
> the average coding coefficient, was most pronounced in ALM, while object location selectivity was
> mainly limited to MM.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; Eb = A[b]["epochs"]
    pb, pa = A[b]["pure"][..., 1], A[b]["pure"][..., 0]
    ab, aa = np.abs(A[b]["coef_sig"][..., 1]), np.abs(A[b]["coef_sig"][..., 0])
    d = Eb["delay"]; w = (t >= 0) & (t < C.GO)
    lick = (pb[ALM].mean(0) > pb[MM].mean(0)) & (ab[ALM].mean(0) > ab[MM].mean(0))
    obj = (pa[MM].mean(0) > pa[ALM].mean(0)) & (aa[MM].mean(0) > aa[ALM].mean(0))
    print(f"{b}: delay bins with ALM above MM on both lick-direction measures {lick[d].sum()} of {d.sum()} "
          f"(fraction alone {(pb[ALM].mean(0) > pb[MM].mean(0))[d].sum()}, mean |beta| alone {(ab[ALM].mean(0) > ab[MM].mean(0))[d].sum()}); "
          f"bins trial start to go with MM above ALM on both object measures {obj[w].sum()} of {w.sum()}")
ab = np.abs(A["T1"]["coef_sig"][..., 1]); keep = cells.f0_min.values > 0
print("Variant, ROIs with F0 > 0 on every trial (T1): delay bins with ALM mean |beta| above MM:",
      int((ab[ALM & keep].mean(0) > ab[MM & keep].mean(0))[E["delay"]].sum()), "of", int(E["delay"].sum()))

**Result: reproduced.** In the delay epoch ALM is above MM on both lick-direction measures in 24 of
42 bins (T1) and 30 of 45 (T2), more than half; from trial start to the go cue MM is above ALM on
both object-location measures in 52 of 59 (T1) and 57 of 63 (T2) bins. The lick-direction part is
carried by the fraction; on mean |beta| alone ALM leads in fewer delay bins (printed).

## figs4-peak-shift

> In two-photon imaging, the peak of activity shifted from MM to ALM over the delay.

In [ ]:
for b in ("T1", "T2"):
    t = A[b]["times"]; Eb = A[b]["epochs"]; D = A[b]["mean_dff"]
    for name, keep in (("all ROIs", np.ones(len(cells), bool)), ("variant, F0 > 0 on every trial", cells.f0_min.values > 0)):
        cA, cM = D[ALM & keep].mean(0), D[MM & keep].mean(0)
        end = cA[Eb["delay"]][-1]
        print(f"{b}, {name}: peak MM {t[cM.argmax()]:.2f} s, ALM {t[cA.argmax()]:.2f} s, MM earlier: {t[cM.argmax()] < t[cA.argmax()]}; "
              f"ALM at the end of the delay {end:.3f}, sample-epoch mean {cA[Eb['sample']].mean():.3f}, max {cA[Eb['sample']].max():.3f}; "
              f"above the mean: {end > cA[Eb['sample']].mean()}, above the max: {end > cA[Eb['sample']].max()}")
t = A["T1"]["times"]; D = A["T1"]["mean_dff"]; keep = cells.f0_min.values > 0
plt.figure(figsize=(5.5, 3))
for lab, m in (("ALM", ALM), ("MM", MM)):
    plt.plot(t, D[m].mean(0), label=f"{lab}, all ROIs")
    plt.plot(t, D[m & keep].mean(0), "--", label=f"{lab}, F0 > 0")
for x in (0, 1.2, 4.2):
    plt.axvline(x, color="k", lw=0.5)
plt.xlabel("time from trial start (s)"); plt.ylabel("mean dF/F"); plt.legend(fontsize=7); plt.show()
for lab, m in (("ALM", ALM), ("MM", MM)):
    print(f"{lab} plotted range: all {D[m].mean(0).min():.3f} to {D[m].mean(0).max():.3f}; F0 > 0 {D[m & keep].mean(0).min():.3f} to {D[m & keep].mean(0).max():.3f}")
pre_go = t < C.GO
print("Variant, peak before the go cue only (T1, F0 > 0):",
      {lab: round(float(t[pre_go][D[m & keep].mean(0)[pre_go].argmax()]), 2) for lab, m in (("ALM", ALM), ("MM", MM))})

**Result: not reproduced.** The mean dF/F of MM ROIs peaks after that of ALM (6.09 against 4.52 s with
T1 bins), so the MM mean does not peak earlier. The ALM mean at the end of the delay (0.212) is above
its sample-epoch level under both readings. Without the ROIs with F0 <= 0, which dominate the MM mean,
MM peaks at 4.80 s, still after ALM. Searching only before the go cue (a variant, not deciding), MM
peaks at 1.51 s and ALM at 4.09 s.

## fig8c-pt-deep-only

> IT neurons were found in both superficial and deep layers, PT neurons only in deep layers; 719 PT
> and 2,631 IT neurons were imaged in three mice.

In [ ]:
n_pt, n_it = int((ctype == "PT").sum()), int((ctype == "IT").sum())
print(f"PT {n_pt} (expected 719 +- 72), IT {n_it} (expected 2631 +- 263); mice with labels: "
      f"{sorted(set(animal[(ctype == 'PT') | (ctype == 'IT')]))}")
print(pd.crosstab([cells.animal, cells.area], cells.cell_type).to_string())
print("Depth of PT and IT neurons cannot be checked: the files hold no plane depth.")

**Result: partially reproduced.** 719 PT and 2,631 IT ROIs in three mice, exactly the paper's counts.
That PT neurons are only in deep layers and IT neurons in both cannot be tested: the files hold no
plane depth.

## fig8d-pt-contra-bias

> During the delay epoch, ALM PT neurons showed a bias toward contralateral (lick right) selectivity.

In [ ]:
rng = np.random.default_rng(81)
pt = ALM & (ctype == "PT")
for b in ("T1", "T2"):
    Eb = A[b]["epochs"]
    v = A[b]["coef_sig"][..., 1][:, Eb["delay"]].mean(1)
    for name, keep in (("all", np.ones(len(cells), bool)), ("variant, F0 > 0", cells.f0_min.values > 0)):
        g_ = pt & keep
        boot = np.array([v[C.joint_resample(animal, plane, [g_], rng)[0]].mean() for _ in range(1000)])
        m = v[g_].mean()
        print(f"{b} {name}: ALM PT n={g_.sum()}, mean beta over the delay {m:.4f}, bootstrap SD {boot.std():.4f}, "
              f"ratio {m / boot.std():.2f}, above 2 SD: {m > 2 * boot.std()}")
v = A["T1"]["coef_sig"][..., 1][:, E["delay"]].mean(1)
print("by animal (T1): " + "; ".join(f"{a} n={int((pt & (animal == a)).sum())} mean {v[pt & (animal == a)].mean():.4f}" for a in np.unique(animal[pt])))
print(f"fraction of ALM PT ROIs with positive delay-mean beta {np.mean(v[pt] > 0):.3f}, negative {np.mean(v[pt] < 0):.3f}")

**Result: not reproduced.** The mean beta of ALM PT ROIs over the delay is positive (lick right,
contralateral), 0.028 with T1 bins, but only 1.71 hierarchical bootstrap SDs above zero (T2 1.56),
short of the 2 SD the criterion requires; without the ROIs with F0 <= 0 it is 1.97 (T1) and 1.86
(T2). The mean is positive in each of the three mice.

## fig8d-pt-earlier-than-it

> In ALM, lick direction selectivity (|beta|) was stronger and earlier in PT than in IT neurons, with an
> onset difference of 1.4 +- 0.6 s (p < 0.01, bootstrap).

Reading I1 (deep ALM IT) needs depth and cannot be run; reading I2 (all ALM IT) is run with bins T1
and T2, baselines O1 and O2 and bootstraps H1 (hierarchical) and H2 (over cells).

In [ ]:
rng = np.random.default_rng(82)
pt, it = ALM & (ctype == "PT"), ALM & (ctype == "IT")
rows = []
for b in ("T1", "T2"):
    t = A[b]["times"]; ab = np.abs(A[b]["coef_sig"][..., 1])
    cp, ci = ab[pt].mean(0), ab[it].mean(0)
    for base in ("O1", "O2"):
        obs = C.onset(ci, t, base) - C.onset(cp, t, base)
        for hier in (True, False):
            d = []
            for _ in range(1000):
                ip, ii = C.joint_resample(animal, plane, [pt, it], rng, hier)
                d.append(C.onset(ab[ii].mean(0), t, base) - C.onset(ab[ip].mean(0), t, base))
            d = np.array(d)
            rows.append({"bins": b, "baseline": base, "bootstrap": "H1" if hier else "H2",
                         "PT_onset": C.onset(cp, t, base), "IT_onset": C.onset(ci, t, base), "IT_minus_PT": obs,
                         "boot_sd": np.nanstd(d), "p_not_earlier": np.mean(~(d > 0)), "no_onset": int(np.isnan(d).sum()),
                         "PT_max": cp.max(), "IT_max": ci.max()})
res = pd.DataFrame(rows)
res["within_1.4_pm_0.6"] = (res.IT_minus_PT - 1.4).abs() <= 0.6
print(res.round(3).to_string(index=False))
t = A["T1"]["times"]; ab = np.abs(A["T1"]["coef_sig"][..., 1])
plt.figure(figsize=(5.5, 3))
plt.plot(t, ab[pt].mean(0), label="ALM PT"); plt.plot(t, ab[it].mean(0), label="ALM IT")
for x in (0, 1.2, 4.2):
    plt.axvline(x, color="k", lw=0.5)
plt.xlabel("time from trial start (s)"); plt.ylabel("mean |beta|"); plt.legend(); plt.show()
print(f"plotted range PT {ab[pt].mean(0).min():.4f} to {ab[pt].mean(0).max():.4f}, IT {ab[it].mean(0).min():.4f} to {ab[it].mean(0).max():.4f}")
print("by animal (T1, O1): " + "; ".join(
    f"{a} PT onset {C.onset(ab[pt & (animal == a)].mean(0), t, 'O1'):.2f}, IT onset {C.onset(ab[it & (animal == a)].mean(0), t, 'O1'):.2f}"
    for a in np.unique(animal[pt])))

**Result: not reproduced.** Mean |beta| is larger in ALM PT than in ALM IT ROIs (maximum 0.229
against 0.084, T1), but its onset is earlier by only 0.2 s with baseline O1 (T1 0.215 s, T2 0.200 s),
and with T1 bins and baseline O2 the IT onset comes first. The bootstrap SD of the difference is 1.36
to 1.93 s and the fraction of resamples in which PT is not earlier is 0.31 to 0.53, so no
combination of readings gives a difference within 1.4 +- 0.6 s or p < 0.01. Reading I1 (deep IT
only) could not be run.

## fig8d-mm-posterior-preferring

> MM neurons were more active in posterior trials, when the pole was within reach of the whiskers,
> than in anterior trials.

In [ ]:
rng = np.random.default_rng(83)
for b in ("T1", "T2"):
    Eb = A[b]["epochs"]
    v = A[b]["coef_sig"][..., 0][:, Eb["sample"]].mean(1)
    for name, keep in (("all", np.ones(len(cells), bool)), ("variant, F0 > 0", cells.f0_min.values > 0)):
        g_ = MM & keep
        boot = np.array([v[C.joint_resample(animal, plane, [g_], rng)[0]].mean() for _ in range(1000)])
        print(f"{b} {name}: MM n={g_.sum()}, mean alpha over the sample epoch {v[g_].mean():.4f}, bootstrap SD {boot.std():.4f}, "
              f"ratio {v[g_].mean() / boot.std():.2f}, above 2 SD: {v[g_].mean() > 2 * boot.std()}")
v = A["T1"]["coef_sig"][..., 0][:, E["sample"]].mean(1)
print("by animal (T1): " + "; ".join(f"{a} n={int((MM & (animal == a)).sum())} mean {v[MM & (animal == a)].mean():.4f}" for a in np.unique(animal[MM])))
sig_a = A["T1"]["sig"][..., 0][:, E["sample"]]
K = A["T1"]["coef"][..., 0][:, E["sample"]]
print(f"MM ROI-bins in the sample epoch with significant alpha: posterior-preferring {(sig_a & (K > 0))[MM].sum()}, anterior-preferring {(sig_a & (K < 0))[MM].sum()}")

**Result: not reproduced.** The mean alpha of MM ROIs over the sample epoch is positive (posterior
preferring), 0.023 with T1 bins, but 1.87 hierarchical bootstrap SDs above zero (T2 1.94), short of
the 2 SD of the criterion (1.70 and 1.63 without the ROIs with F0 <= 0). It is positive in all five
mice with MM planes, and significant MM ROI-bins in the sample epoch prefer the posterior pole 1,912
times against 747.

## fig8d-error-outnumber-correct

> Overall, neurons preferring error trials outnumbered those preferring correct trials.

In [ ]:
rng = np.random.default_rng(84)
for b in ("T1", "T2"):
    t = A[b]["times"]; w = (t >= C.GO) & (t <= C.GO + 2.5 + 1e-9)
    pg = A[b]["pure"][:, w, 2]; P = A[b]["p"][:, w, 2]; K = A[b]["coef"][:, w, 2]
    has = pg.any(1)
    best = np.where(pg, P, 2).argmin(1)
    sgn = np.sign(K[np.arange(len(K)), best])[has]
    n_err, n_cor = int((sgn < 0).sum()), int((sgn > 0).sum())
    bt = stats.binomtest(n_err, n_err + n_cor, 0.5)
    v = A[b]["coef_sig"][:, w, 2].mean(1)
    allc = np.ones(len(cells), bool)
    boot = np.array([v[C.joint_resample(animal, plane, [allc], rng)[0]].mean() for _ in range(1000)])
    print(f"{b}: (a) error-preferring {n_err}, correct-preferring {n_cor}, binomial p = {bt.pvalue:.2e}; "
          f"(b) mean gamma {v.mean():.4f}, bootstrap SD {boot.std():.4f}, ratio {v.mean() / boot.std():.2f}, below -2 SD: {v.mean() < -2 * boot.std()}")
t = A["T1"]["times"]; w = (t >= C.GO) & (t <= C.GO + 2.5 + 1e-9)
v = A["T1"]["coef_sig"][:, w, 2].mean(1)
print("by animal (T1), mean gamma: " + "; ".join(f"{a} {v[animal == a].mean():.4f}" for a in np.unique(animal)))
keep = cells.f0_min.values > 0
print(f"variant, F0 > 0 (T1): mean gamma {v[keep].mean():.4f}")

**Result: partially reproduced.** Reading (a) finds error preference: among ROIs purely outcome selective
after the go cue, 2,825 prefer error trials and 2,004 correct trials (T1, binomial p = 2.8e-32), and
T2 agrees (2,730 against 1,933). Reading (b) does not reach its bound: the mean gamma over all ROIs is
negative (-0.0082, T1), and in every mouse, but 1.81 bootstrap SDs from zero (T2 1.71). The criterion requires both readings, so the claim
holds only in part.

## alm-pt-it-lick-fraction-below-20

> Even in ALM, less than 20% of PT or IT cells displayed lick direction selectivity at any given time,
> and the onset timing was variable.

In [ ]:
pt, it = ALM & (ctype == "PT"), ALM & (ctype == "IT")
for b in ("T1", "T2"):
    t = A[b]["times"]
    for name, X in (("(a) purely lick-direction", A[b]["pure"][..., 1]), ("(b) beta significant", A[b]["sig"][..., 1])):
        fp, fi = X[pt].mean(0), X[it].mean(0)
        print(f"{b} {name}: max PT {100 * fp.max():.1f}% at {t[fp.argmax()]:.2f} s, max IT {100 * fi.max():.1f}% at {t[fi.argmax()]:.2f} s; "
              f"both below 22%: {fp.max() < 0.22 and fi.max() < 0.22}")
X = A["T1"]["sig"][..., 1]
print("by animal (T1, reading b), max PT and IT: " + "; ".join(
    f"{a} {100 * X[pt & (animal == a)].mean(0).max():.1f}% / {100 * X[it & (animal == a)].mean(0).max():.1f}%" for a in np.unique(animal[pt])))

**Result: reproduced.** The largest per-bin fraction of lick-direction-selective ALM PT ROIs is 12.7%
(purely selective) or 15.6% (beta significant), and of ALM IT ROIs 3.3% or 6.0% (T1); T2 gives
similar values (16.1% at most). All stay below 22%, also in each mouse (at most 19.8%). The variability of onset is not judged.

## li2015-tagged-units-link

> The tagged units of DANDI:000010 (Li et al. 2015) are the 45 PT and 27 IT cells the paper compares
> its imaging with.

In [ ]:
li = C.li2015_tagged_units()
types = collections.Counter(t for r in li for t in r["cell_type"])
print("files:", len(li), " files with units:", sum(len(r["cell_type"]) > 0 for r in li))
print("cell_type counts over all units:", dict(types))
print("PT units by genotype:", dict(collections.Counter(r["genotype"] for r in li for t in r["cell_type"] if t == "PT")))
print("IT units by genotype:", dict(collections.Counter(r["genotype"] for r in li for t in r["cell_type"] if t == "IT")))

**Result: reproduced.** DANDI:000010 version 0.220126.1905 holds 45 units labelled PT and 27 labelled
IT, the counts the paper gives for the electrophysiology (45/27). The units are matched by count and
label only; no identifier links them to the paper's analysis.

## Summary

| Claim | Outcome |
|---|---|
| early-lick-fraction | Reproduced |
| imaging-sample | Partially reproduced (locations not identifiable) |
| trials-per-type | Reproduced |
| neuron-counts | Reproduced |
| fraction-selective-alm-mm | Not reproduced |
| figs5-single-coefficient | Reproduced |
| fig4a-lick-direction-epochs | Reproduced |
| fig4b-object-location-memory | Reproduced |
| fig4c-outcome-after-lick | Not reproduced |
| fig5-mixed-response-epoch | Reproduced |
| fig6-object-location-mm | Reproduced |
| fig6-lick-direction-deep-alm-first | Blocked: metadata insufficient (no depth) |
| fig6-lick-direction-spread | Partially reproduced (depth part not testable) |
| fig6-outcome-response-uniform | Not reproduced |
| fig6h-preparatory-alm-object-mm | Reproduced |
| figs9-photoinhibition-alm-mm | Blocked: data missing |
| widefield-mm-to-alm | Blocked: data missing |
| widefield-contralateral-bilateral | Blocked: data missing |
| figs4-peak-shift | Not reproduced |
| fig8c-pt-deep-only | Partially reproduced (counts only) |
| fig8d-alm-deep-lick-direction | Blocked: metadata insufficient (no depth) |
| fig8d-pt-contra-bias | Not reproduced |
| fig8d-pt-earlier-than-it | Not reproduced |
| ephys-pt-it-onset-not-detected | Blocked: methods underspecified (question asked) |
| fig8d-mm-object-location | Blocked: metadata insufficient (no depth) |
| fig8d-mm-posterior-preferring | Not reproduced |
| fig8d-error-outnumber-correct | Partially reproduced |
| fig8ef-area-explains-variance | Blocked: metadata insufficient (no depth) |
| ephys-selective-counts | Blocked: the silicon probe data were never made public |
| fig7d-ephys-area-distribution | Blocked: the silicon probe data were never made public |
| fig1c-whisking-similar | Blocked: data missing |
| alm-pt-it-lick-fraction-below-20 | Reproduced |
| li2015-tagged-units-link (link) | Reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The time bins (frames on one grid, or 66.7 ms by
interpolation), the handling of baselines F0 <= 0 (none), the onset rule (first bin over 6 SD) and the
bootstrap details are read from the claims file's conventions, which leave them open or fix them
without the paper's code; several claims use thresholds the claims file fixed itself (2 bootstrap SD,
more than half of bins). Area comes from the files' label. Readings that need depth or coordinates
could not be run. fig8d-pt-contra-bias and fig8d-mm-posterior-preferring, and the bootstrap part of
fig8d-error-outnumber-correct, find the stated direction at 1.56 to 1.94 bootstrap SDs, short of the
2 SD bound the claims file fixed. The per-bin selectivity fractions are low (ALM purely
lick-direction selective at most 6.2%), and the rules that count neurons selective in any bin count
many chance bins.

**What was found about the dataset.** Each of the 105 planes is deposited twice, under different
identifiers. `ImagingPlane.location` says ALM for every plane, also the MM ones. The plane depth,
the cells' coordinates, the imaging location within a mouse and the lick times are not in the files,
nor are the wide-field imaging, the whisker tracking, the photoinhibition data and the silicon probe
recordings; the CRCNS DOI the paper gives for its data is an unfilled placeholder. The `included`
column marks 8,655 ROIs without saying why. After neuropil subtraction, 1,262 ROIs have a zero or
negative baseline on some trial. Early-lick trials carry no pole position. The counts of planes,
ROIs, areas, PT and IT cells and trials agree with the paper.

In [ ]:
print(f"total runtime {(time.time() - started) / 60:.1f} min")